# Duhamel Transport Beta Guide 03: Bring Your Own Liouvillian

This notebook is a tutorial template for a new user-defined model. It shows the full workflow:

1. Define Hilbert-space operators.
2. Write `H(params)`, `c_ops(params)`, and `mu_components(params)`.
3. Wrap those pieces as a `LiouvilleModel`.
4. Choose a reference model and selected control coordinates `lambda`.
5. Retain phase-matched pathway bases.
6. Compute Liouvillian derivative tensors.
7. Build the Duhamel transport matrix `T(lambda)`.
8. Compute final reference, target, and transported signals.

The example model is a compact three-level V system, but the same cells are meant to be edited in place for a new molecule, aggregate, spin system, or effective model.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.linalg import block_diag

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import Liouville_transport_general as general
from demo_dimer_duhamel_transport import exact_transport_matrix_2d, normalize_to_unit_power
from fit_v_system_transport_signal import (
    duhamel_transport_expansion_2d,
    pathway_signal_grid,
    relative_error,
    transported_signal,
)


## 1. Define elementary Hilbert-space operators

The code uses column-stacked density matrices internally, but user-facing model construction is intentionally Hilbert-space first. This mirrors the usual QuTiP style: define basis states, projectors, transition operators, Hamiltonian, and collapse operators.


In [ ]:
def ket(index, dim):
    out = np.zeros((dim, 1), dtype=complex)
    out[index, 0] = 1.0
    return out


def projector(index, dim):
    state = ket(index, dim)
    return state @ state.conj().T


def op(i, j, dim):
    return ket(i, dim) @ ket(j, dim).conj().T


dim = 3
states = {"g": 0, "b": 1, "d": 2}
P = {name: projector(index, dim) for name, index in states.items()}
X = {
    f"{left}_{right}": op(states[left], states[right], dim)
    for left in states
    for right in states
}


## 2. Write the model-specific functions

For a new model, this is the main block to replace.

Here the V system has a ground state `|g>`, a bright excited state `|b>`, and a dark excited state `|d>`. The Hamiltonian includes coherent bright-dark exchange `J_bd`. The Lindblad operators include population relaxation, pure dephasing, and detailed-balance incoherent transfer between `|b>` and `|d>`.


In [ ]:
def detailed_balance_transfer_rates(params):
    """Return rates k_{b->d}, k_{d->b} with k_B = 1."""

    delta = params["omega_d"] - params["omega_b"]
    beta = params.get("beta_bd", 1.0 / params["temperature"])
    rate = params["k_bd"]
    if np.isinf(beta):
        if delta > 0:
            return 0.0, rate
        if delta < 0:
            return rate, 0.0
        return rate, rate
    return (
        rate * np.exp(-0.5 * beta * delta),
        rate * np.exp(+0.5 * beta * delta),
    )


def H(params):
    return (
        params["omega_b"] * P["b"]
        + params["omega_d"] * P["d"]
        + params["J_bd"] * (X["b_d"] + X["d_b"])
    )


def c_ops(params):
    collapse_ops = []
    k_b_to_d, k_d_to_b = detailed_balance_transfer_rates(params)

    if params["gamma_b"] > 0:
        collapse_ops.append(np.sqrt(params["gamma_b"]) * X["g_b"])
    if params["gamma_d"] > 0:
        collapse_ops.append(np.sqrt(params["gamma_d"]) * X["g_d"])
    if params["gamma_phi_b"] > 0:
        collapse_ops.append(np.sqrt(params["gamma_phi_b"]) * P["b"])
    if params["gamma_phi_d"] > 0:
        collapse_ops.append(np.sqrt(params["gamma_phi_d"]) * P["d"])
    if k_b_to_d > 0:
        collapse_ops.append(np.sqrt(k_b_to_d) * X["d_b"])
    if k_d_to_b > 0:
        collapse_ops.append(np.sqrt(k_d_to_b) * X["b_d"])
    return tuple(collapse_ops)


def mu_components(params):
    mu_plus = params["mu_gb"] * X["b_g"] + params["mu_gd"] * X["d_g"]
    mu_minus = mu_plus.conj().T
    return mu_plus, mu_minus


def rho_initial(_params):
    return general.vec(P["g"])


## 3. Wrap the functions as a `LiouvilleModel`

`LiouvilleModel.from_hamiltonian(...)` assembles the Liouvillian from `H` and `c_ops` using the Lindblad equation

\[
\dot\rho = -i[H,\rho] + \sum_k \left(C_k \rho C_k^\dagger - \frac{1}{2}\{C_k^\dagger C_k,\rho\}\right).
\]


In [ ]:
model = general.LiouvilleModel.from_hamiltonian(
    dimension=dim,
    hamiltonian=H,
    collapse_ops=c_ops,
    initial_state=rho_initial,
    transition_dipole=lambda params: sum(mu_components(params)),
    dipole_components=mu_components,
)

reference_params = {
    "omega_b": 1.00,
    "omega_d": 1.18,
    "J_bd": 0.0,
    "gamma_b": 0.018,
    "gamma_d": 0.010,
    "gamma_phi_b": 0.012,
    "gamma_phi_d": 0.012,
    "k_bd": 0.004,
    "temperature": 0.4,
    "mu_gb": 1.0,
    "mu_gd": 0.15,
}

parameter_map = general.ParameterMap(
    reference_params=reference_params,
    controls=(
        general.ControlSpec("J_bd", reference_value=reference_params["J_bd"], scale=0.035),
        general.ControlSpec("k_bd", reference_value=reference_params["k_bd"], scale=0.010),
    ),
)

target_lambda = np.array([0.45, 0.30])
print("reference parameters:")
for key, value in parameter_map.params(parameter_map.zero()).items():
    print(f"  {key}: {value:g}")
print("\ntarget lambda:", target_lambda)
print("target physical displacements:", parameter_map.physical_displacements(target_lambda))
print("target selected parameters:", {name: parameter_map.params(target_lambda)[name] for name in ("J_bd", "k_bd")})


## 4. Sanity-check the Liouvillian and dipole

This cell catches the most common model-construction mistakes: wrong operator dimensions, a malformed initial density matrix, or a dipole operator that does not match the Hilbert-space dimension.


In [ ]:
params0 = parameter_map.params(parameter_map.zero())
L0 = model.L(params0)
rho0 = model.rho0(params0)
mu0 = model.mu(params0)

print("Liouvillian shape:", L0.shape)
print("rho0 shape:", rho0.shape)
print("mu shape:", mu0.shape)
print("trace rho0:", np.trace(general.mat(rho0, dim)))
print("number of collapse operators:", len(c_ops(params0)))


## 5. Retain independent phase-matched pathway bases

The retained pathway states at the reference model define the `lambda = 0` basis. We build RP and NRP separately so the transport fit respects wave-vector conservation.


In [ ]:
omega = np.linspace(0.8, 1.4, 18)
tau2 = 1.0
eta = 4e-3
sectors = ("RP", "NRP")

sector_pathways = {}
for sector in sectors:
    candidates = general.canonical_pathways((sector,))
    pathways, residuals = general.independent_canonical_pathways(
        model,
        parameter_map,
        omega,
        candidates,
        stage="after_G3",
        tau2=tau2,
        eta=eta,
        rtol=1e-9,
    )
    sector_pathways[sector] = pathways
    print(f"{sector}: retained {len(pathways)} of {len(candidates)} candidate pathways")
    for idx, pathway in enumerate(pathways):
        print(f"  {idx}: {pathway.label}")


## 6. Compute Liouvillian derivative tensors

This tutorial uses the portable finite-difference derivative backend, which works for any NumPy model. These derivative tensors are still inserted through the Duhamel formulas when building pathway-state derivatives.

For a production autodiff model, replace this cell with `Liouville_autodiff.jax_liouvillian_derivatives(...)` or `jax_liouvillian_derivatives_from_lambda(...)` after writing a JAX-compatible Liouvillian.


In [ ]:
liouvillian_derivatives = general.finite_difference_liouvillian_derivatives(
    model,
    parameter_map,
    step=1e-4,
)

print("derivative keys:", sorted(liouvillian_derivatives))
for key in sorted(liouvillian_derivatives):
    print(f"{key or 'L0'}: norm={np.linalg.norm(liouvillian_derivatives[key]):.6e}")


## 7. Build transport expansions and exact projected transport matrices

For each sector we compute:

- reference pathway signals `S_b(0)`,
- exact target pathway signals `S_a(lambda)`,
- the Duhamel/Taylor transport expansion `T(lambda)`,
- the exact projected transport matrix for diagnostics.


In [ ]:
sector_results = []
for sector in sectors:
    pathways = sector_pathways[sector]
    reference_pathway_signals = pathway_signal_grid(
        model,
        parameter_map,
        parameter_map.zero(),
        omega,
        omega,
        pathways,
        tau2=tau2,
        eta=eta,
    )
    target_pathway_signals = pathway_signal_grid(
        model,
        parameter_map,
        target_lambda,
        omega,
        omega,
        pathways,
        tau2=tau2,
        eta=eta,
    )
    expansion = duhamel_transport_expansion_2d(
        model,
        parameter_map,
        omega,
        omega,
        pathways,
        liouvillian_derivatives=liouvillian_derivatives,
        tau2=tau2,
        eta=eta,
        n_quad=6,
        max_order=2,
        rcond=1e-11,
    )
    T_exact = exact_transport_matrix_2d(
        model,
        parameter_map,
        target_lambda,
        omega,
        omega,
        pathways,
        tau2=tau2,
        eta=eta,
        rcond=1e-11,
    )
    sector_results.append(
        {
            "sector": sector,
            "pathways": pathways,
            "reference_pathway_signals": reference_pathway_signals,
            "target_pathway_signals": target_pathway_signals,
            "expansion": expansion,
            "T_order2": expansion.evaluate(target_lambda, order=2),
            "T_exact": T_exact,
            "reconstructed_order1": transported_signal(reference_pathway_signals, expansion, target_lambda, order=1),
            "reconstructed_order2": transported_signal(reference_pathway_signals, expansion, target_lambda, order=2),
        }
    )
    print(f"finished {sector}")


## 8. Compare target and transported final signals

The final signal is obtained after transport by summing the transported pathway contributions. In a different observable convention, this is where one would trace against the chosen detection operator.


In [ ]:
reference_signal = sum(np.sum(item["reference_pathway_signals"], axis=1) for item in sector_results)
target_signal = sum(np.sum(item["target_pathway_signals"], axis=1) for item in sector_results)
reconstructed_order1 = sum(item["reconstructed_order1"] for item in sector_results)
reconstructed_order2 = sum(item["reconstructed_order2"] for item in sector_results)
exact_projected_signal = sum(
    item["reference_pathway_signals"] @ (item["T_exact"].T @ np.ones(item["T_exact"].shape[0], dtype=complex))
    for item in sector_results
)

print("relative errors against exact target signal")
print("  exact projected T:", relative_error(target_signal, exact_projected_signal))
print("  order 1 Duhamel: ", relative_error(target_signal, reconstructed_order1))
print("  order 2 Duhamel: ", relative_error(target_signal, reconstructed_order2))


## 9. Inspect the transport matrix

The exact projected transport matrix is a diagnostic reference. The order-2 Duhamel matrix is the truncated local expansion. White entries in the real-part plot are close to zero; colored off-diagonal entries indicate reference-pathway mixing.


In [ ]:
T_exact = block_diag(*[item["T_exact"] for item in sector_results])
T_order2 = block_diag(*[item["T_order2"] for item in sector_results])
labels = []
for item in sector_results:
    labels.extend([f"{item['sector']}:{idx}" for idx, _ in enumerate(item["pathways"])])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], T_exact.real, "exact projected T"),
    (axes[1], T_order2.real, "order-2 Duhamel T"),
    (axes[2], (T_order2 - T_exact).real, "order-2 minus exact"),
]:
    vmax = max(abs(matrix.min()), abs(matrix.max()), 1e-12)
    image = ax.imshow(matrix, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax))
    ax.set_title(title)
    ax.set_xlabel("reference pathway index b")
    ax.set_ylabel("transported pathway index a")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(image, ax=ax, shrink=0.78)
plt.show()

print("relative matrix error:", np.linalg.norm(T_order2 - T_exact) / max(np.linalg.norm(T_exact), 1e-14))


## 10. Plot final spectra

The target and transported spectra are normalized to the target integrated power before plotting so residual amplitudes are visually meaningful.


In [ ]:
def normalize_like_target(signal, target_norm):
    normalized, _ = normalize_to_unit_power(signal, omega, omega, norm=target_norm)
    return normalized

_, target_power = normalize_to_unit_power(target_signal, omega, omega)
target_plot = normalize_like_target(target_signal, target_power)
reconstructed_plot = normalize_like_target(reconstructed_order2, target_power)
residual_plot = normalize_like_target(reconstructed_order2 - target_signal, target_power)


def contour(ax, signal, title, component="real"):
    grid = signal.reshape(len(omega), len(omega))
    if component == "abs":
        values = np.abs(grid)
        levels = np.linspace(0.0, values.max(), 18)
        image = ax.contourf(omega, omega, values, levels=levels, cmap="magma")
    else:
        values = np.real(grid) if component == "real" else np.imag(grid)
        vmax = max(abs(values.min()), abs(values.max()), 1e-12)
        levels = np.linspace(-vmax, vmax, 21)
        image = ax.contourf(omega, omega, values, levels=levels, cmap="RdBu_r")
    ax.set_title(title)
    ax.set_xlabel(r"$\omega_1$ / eV")
    ax.set_ylabel(r"$\omega_3$ / eV")
    return image

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for col, (name, signal) in enumerate([
    ("target", target_plot),
    ("transported", reconstructed_plot),
    ("residual", residual_plot),
]):
    image = contour(axes[0, col], signal, f"{name}: real", "real")
    fig.colorbar(image, ax=axes[0, col], shrink=0.85)
    image = contour(axes[1, col], signal, f"{name}: abs", "abs")
    fig.colorbar(image, ax=axes[1, col], shrink=0.85)
plt.show()


## 11. Adapting this notebook to a new project

To use this notebook for a new model, edit the model-definition cells in this order:

1. Change `dim`, basis labels, and elementary operators.
2. Replace `H(params)` with your Hamiltonian.
3. Replace `c_ops(params)` with your Lindblad collapse operators.
4. Replace `mu_components(params)` with the perturbation/dipole components used by the response pathways.
5. Update `reference_params` with the best-guess model.
6. Update `ParameterMap.controls` to select the parameters included in the expansion.
7. Choose `target_lambda` and the spectral grid.

Everything below those cells should continue to run without model-specific edits. For larger systems, start with a small frequency grid and increase `n_quad` and `n_omega` only after the pathway labels, matrix conditioning, and signal normalization look reasonable.
